# 2311402 - HEX - Lab4

## Sentiment Analysis Exercise  
The goal of the Sentiment Analysis (SA) task is to classify the sentiment within a piece of text. This sentiment can be either binary (positive-negative) or multi-class (a rating scale from 1 to 5).
 
Data: http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Digital_Music_5.json.gz
 
Steps to Solve This Exercise
1. Load data into a pandas DataFrame
2. reprocessing & Vectorizing 
3. Building Model
4. Evaluation

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import seaborn
import os, shutil
import zipfile, gzip, urllib, json

## Load the data file into a pandas DataFrame

In [ ]:
# Download file
data_dir = "./data"
url = "http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Digital_Music_5.json.gz"
data_path = os.path.join(data_dir, "reviews_Digital_Music_5.json.gz")

urllib.request.urlretrieve(url, data_path)  

In [ ]:
# Extract to the data file
data_dir = "./data"

def unzip_file(path, destination, delete_zip=False):
    if path.endswith(".zip"):
        with zipfile.ZipFile(path, 'r') as zip_ref:
            zip_ref.extractall(path)
        print(f"Extracted zip to {destination}")
    elif path.endswith(".gz"):
        out_file = os.path.join(
            destination, 
            os.path.basename(path).replace(".gz", "")
        )
        with gzip.open(path, 'rb') as f_in:
            with open(out_file, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        print(f"Extracted gzip to {out_file}")
    else: 
        print(f"This file format is not accepted")
        return 
    if delete_zip:
        os.remove(path)
        print("Removed compressed file")
        return        
    print("Only unzip file")
    
unzip_file(data_path, data_dir, True)

Extracted gzip to ./data/reviews_Digital_Music_5.json
Removed compressed file


In [16]:
# Load the data
data_path = f"{data_dir}/reviews_Digital_Music_5.json"
def load_data(file_path):
    """Load JSON lines data from file or gz file"""
    data = []
    # Load the JSON lines file
    if os.path.exists(file_path):
        print(f"Loading data from {file_path}...")
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                data.append(json.loads(line))
    else:
        print(f"File not found: {file_path}")
        print("Please ensure the data file is in the ./data directory")
        return None
    
    df = pd.DataFrame(data)
    print(f"Loaded {len(df)} reviews")
    return df

# Load the data
df = load_data(data_path)

Loading data from ./data/reviews_Digital_Music_5.json...
Loaded 64706 reviews


In [17]:
# Display basic info
print("\nDataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nFirst few rows:")
print(df.head(2))
print("\nRating distribution:")
print(df['overall'].value_counts().sort_index())


Dataset shape: (64706, 9)

Column names: ['reviewerID', 'asin', 'reviewerName', 'helpful', 'reviewText', 'overall', 'summary', 'unixReviewTime', 'reviewTime']

First few rows:
       reviewerID        asin          reviewerName helpful  \
0  A3EBHHCZO6V2A4  5555991584  Amaranth "music fan"  [3, 3]   
1   AZPWAXJG9OJXV  5555991584             bethtexas  [0, 0]   

                                          reviewText  overall  \
0  It's hard to believe "Memory of Trees" came ou...      5.0   
1  A clasically-styled and introverted album, Mem...      5.0   

                    summary  unixReviewTime   reviewTime  
0   Enya's last great album      1158019200  09 12, 2006  
1  Enya at her most elegant       991526400   06 3, 2001  

Rating distribution:
overall
1.0     2791
2.0     3010
3.0     6789
4.0    16536
5.0    35580
Name: count, dtype: int64


## Preprocessing data

In [18]:
# Preprocessing data
import re
def preprocess_text(text):
    if pd.isna(text):
        return ""
    
    # Convert to lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Handle common contractions (important for sentiment)
    contractions = {
        "'nt":"not",
        "n't": " not",
        "'re": " are",
        "'s": " is",
        "'d": " would",
        "'ll": " will",
        "'ve": " have",
        "'m": " am"
    }
    for contraction, expansion in contractions.items():
        text = text.replace(contraction, expansion)
    
    # Keep only letters and spaces (remove numbers, punctuation)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Remove very short words (length < 2)
    text = ' '.join([word for word in text.split() if len(word) > 1])
    
    return text

In [19]:
# Apply preprocessing
df['cleaned_review'] = df['reviewText'].apply(preprocess_text)

# Remove empty reviews
df = df[df['cleaned_review'] != '']

print(f"Reviews after cleaning: {len(df)}")

# Create binary sentiment labels (1-3: Negative, 4-5: Positive)
df['sentiment_binary'] = df['overall'].apply(lambda x: 'positive' if x >= 4 else 'negative')
print("\nBinary sentiment distribution:")
print(df['sentiment_binary'].value_counts())

# Multi-class labels (keep original ratings 1-5)
df['sentiment_multiclass'] = df['overall'].astype(int)

Reviews after cleaning: 64705

Binary sentiment distribution:
sentiment_binary
positive    52116
negative    12589
Name: count, dtype: int64


## Feature Extraction

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
# Split data for binary classification
X = df['cleaned_review']
y_binary = df['sentiment_binary']
y_multiclass = df['sentiment_multiclass']

# Train-test split
X_train, X_test, y_train_bin, y_test_bin = train_test_split(
    X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

_, _, y_train_multi, y_test_multi = train_test_split(
    X, y_multiclass, test_size=0.2, random_state=42, stratify=y_multiclass
)

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2),
    stop_words='english'
)

In [22]:
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Training set size: {X_train_tfidf.shape}")
print(f"Testing set size: {X_test_tfidf.shape}")

Training set size: (51764, 5000)
Testing set size: (12941, 5000)
